# 数据加载与全流程清洗

本 Notebook 用于记录酒店预订行为与取消预测项目的数据加载、质量检查和基础清洗流程。

说明：字段是原始数据列；特征是后续建模使用的输入变量。本阶段新增的 `total_guests`、`total_stays`、`agent_missing`、`company_missing` 主要用于清洗和后续判断，是否作为最终建模特征还需要结合业务含义和数据泄漏风险确认。

## 1. 导入工具库与路径配置

In [ ]:
# 配置项目路径和常用显示选项，保证 notebook 在项目根目录或 notebooks 目录下都能运行。
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "hotel_bookings_updated_2024.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "hotel_bookings_cleaned.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

## 2. 读取原始数据

使用相对路径读取 `data/raw/hotel_bookings_updated_2024.csv`。

In [64]:
# 读取原始 CSV 前先检查文件是否存在，避免路径错误时继续执行。
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"数据文件不存在：{RAW_DATA_PATH}")

df = pd.read_csv(RAW_DATA_PATH)
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,city
0,Resort Hotel - Chandigarh,0,342,2024,July,30,27,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2024-07-27 22:16:40.916332324,Chandigarh
1,Resort Hotel - Mumbai,0,737,2024,April,17,28,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2024-04-28 21:56:21.507509066,Mumbai
2,Resort Hotel - Delhi,0,7,2024,September,37,10,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2024-09-10 03:46:25.734029096,Delhi
3,Resort Hotel - Kolkata,0,13,2024,August,33,14,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2024-08-14 18:07:10.049669568,Kolkata
4,Resort Hotel - Lucknow,0,14,2024,September,37,14,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2024-09-14 14:27:32.473846000,Lucknow


## 3. 数据基础概览

In [65]:
print(f"行数：{df.shape[0]:,}")
print(f"列数：{df.shape[1]:,}")

行数：119,390
列数：33


In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 33 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

In [67]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
hotel,119390,30,City Hotel - Ahmedabad,5405,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_canceled,119390.0,NaN,NaN,NaN,0.370416,0.482918,0.0,0.0,0.0,1.0,1.0
lead_time,119390.0,NaN,NaN,NaN,104.011416,106.863097,0.0,18.0,69.0,160.0,737.0
arrival_date_year,119390.0,NaN,NaN,NaN,2024.0,0.0,2024.0,2024.0,2024.0,2024.0,2024.0
arrival_date_month,119390,12,October,10349,NaN,NaN,NaN,NaN,NaN,NaN,NaN
arrival_date_week_number,119390.0,NaN,NaN,NaN,26.375835,15.021596,1.0,13.0,26.0,39.0,52.0
arrival_date_day_of_month,119390.0,NaN,NaN,NaN,15.723394,8.805079,1.0,8.0,16.0,23.0,31.0
stays_in_weekend_nights,119390.0,NaN,NaN,NaN,0.927599,0.998613,0.0,0.0,1.0,2.0,19.0
stays_in_week_nights,119390.0,NaN,NaN,NaN,2.500302,1.908286,0.0,1.0,2.0,3.0,50.0
adults,119390.0,NaN,NaN,NaN,1.856403,0.579261,0.0,2.0,2.0,2.0,55.0


## 4. 数据质量检查

In [68]:
# 调用 src 中的数据质量检查函数，统一检查字段完整性和缺失情况。
from data.data_preprocessing import build_quality_report, validate_schema

validate_schema(df)
quality_report = build_quality_report(df)
quality_report.head(15)

发现项目说明之外的字段，暂不删除：['city']


,dtype,missing_count,missing_ratio,unique_count
company,float64,112593,0.943069,352
agent,float64,16340,0.136862,333
country,str,488,0.004087,177
children,float64,4,0.000034,5
hotel,str,0,0.000000,30
is_canceled,int64,0,0.000000,2
lead_time,int64,0,0.000000,479
arrival_date_year,int64,0,0.000000,1
arrival_date_month,str,0,0.000000,12
arrival_date_week_number,int64,0,0.000000,52


## 5. 目标变量检查

项目说明文件中明确写明 `is_canceled` 是目标变量，含义为是否取消预订，`0=未取消`，`1=取消`。本节检查目标变量是否存在、类别分布是否合理，并判断任务类型。

In [69]:
# 确认目标变量存在，并查看取消与未取消订单的分布。
TARGET = "is_canceled"

if TARGET not in df.columns:
    raise ValueError(f"目标字段 {TARGET} 不在数据集中。")

target_counts = df[TARGET].value_counts(dropna=False).rename("count")
target_ratios = df[TARGET].value_counts(normalize=True, dropna=False).rename("ratio")
target_summary = pd.concat([target_counts, target_ratios], axis=1)
target_summary["ratio"] = target_summary["ratio"].map(lambda x: f"{x:.2%}")
target_summary


,count,ratio
is_canceled,,
0,75166,62.96%
1,44224,37.04%


## 6. 字段完整性与重复性检查

本节检查当前数据字段是否与项目说明一致，并检查完全重复行与可能的唯一 ID 字段。当前数据比项目说明多出 `city` 字段；由于项目说明没有解释该字段，本阶段只记录为额外字段，不强行解释其业务含义。

In [70]:
# 对比项目说明中的字段和实际数据字段，记录缺失字段与额外字段。
from data.data_preprocessing import EXPECTED_COLUMNS

expected_columns = EXPECTED_COLUMNS
actual_columns = df.columns.tolist()

missing_columns = [col for col in expected_columns if col not in actual_columns]
extra_columns = [col for col in actual_columns if col not in expected_columns]

print(f"项目说明字段数量：{len(expected_columns)}")
print(f"实际数据字段数量：{len(actual_columns)}")
print(f"缺少字段：{missing_columns}")
print(f"额外字段：{extra_columns}")


项目说明字段数量：32
实际数据字段数量：33
缺少字段：[]
额外字段：['city']


In [71]:
# 检查完全重复行，并尝试识别是否存在可用于去重的唯一 ID 字段。
duplicate_rows = df.duplicated().sum()
print(f"完全重复行数量：{duplicate_rows:,}")

id_candidates = [
    col for col in df.columns
    if col.lower() == "id" or col.lower().endswith("_id") or "id" in col.lower()
]

print(f"可能的 ID 字段：{id_candidates}")
if id_candidates:
    for col in id_candidates:
        print(f"{col} 是否唯一：{df[col].is_unique}")
else:
    print("当前数据中没有发现明确的唯一 ID 字段。")


完全重复行数量：0
可能的 ID 字段：[]
当前数据中没有发现明确的唯一 ID 字段。


## 7. 多维度数据清洗

本阶段围绕项目要求执行多维度数据清洗，主要包括：

1. 缺失值处理：对 `children`、`agent`、`company` 使用 0 进行缺失值填补。
2. 异常值处理：删除 `adr` 小于 0 的记录，并使用 IQR 规则处理 `adr` 和入住天数异常值。
3. 数据去重：当前数据没有明确唯一预订 ID，因此只检查完全重复行。
4. 时间格式统一：将 `reservation_status_date` 转换为 datetime，并构造 `arrival_date`。

### 7.1 缺失值处理

本节针对项目说明中要求关注的 `children`、`agent`、`company` 三个字段进行缺失值处理。

- `children`：儿童数量，缺失数量很少，本阶段使用 0 填补。
- `agent`：代理商 ID，属于匿名 ID 字段，缺失值本阶段使用 0 填补。
- `company`：公司 ID，属于匿名 ID 字段，缺失比例较高，本阶段使用 0 填补。

`agent` 和 `company` 的缺失可能本身具有业务含义，但目前缺少更明确的数据字典支持，因此本阶段只做基础缺失填补；是否新增缺失标记字段放到后续特征工程阶段判断。

In [72]:
# 对指定缺失字段采用 0 填充，表示无儿童、无代理商或无公司信息。
missing_fill_cols = ["children", "agent", "company"]

missing_before = df[missing_fill_cols].isna().sum().rename("missing_before")

df_cleaned_missing = df.copy()
df_cleaned_missing[missing_fill_cols] = df_cleaned_missing[missing_fill_cols].fillna(0)

missing_after = df_cleaned_missing[missing_fill_cols].isna().sum().rename("missing_after")
pd.concat([missing_before, missing_after], axis=1)


,missing_before,missing_after
children,4,0
agent,16340,0
company,112593,0


### 7.2 异常值处理

本节先处理 `adr` 房价异常。入住晚数和住客数量是否作为异常值处理，需要结合业务场景进一步确认，暂不在本节删除。

#### 7.2.1 `adr` 房价异常处理

`adr` 表示平均每日房价。根据业务逻辑，房价不应为负；对于极端高值，本阶段使用 IQR 法则识别统计异常值。IQR 规则不是只保留 Q1 到 Q3 中间 50% 的数据，而是使用 `[Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]` 作为合理范围。由于房价不能为负，下界限制为 0。

In [73]:
# 使用 IQR 法则计算 ADR 房价异常边界，同时保留业务规则：房价不能为负。
df_cleaned_adr = df_cleaned_missing.copy()

q1 = df_cleaned_adr["adr"].quantile(0.25)
q3 = df_cleaned_adr["adr"].quantile(0.75)
iqr = q3 - q1
adr_lower = max(0, q1 - 1.5 * iqr)
adr_upper = q3 + 1.5 * iqr

adr_summary = pd.Series({
    "q1": q1,
    "q3": q3,
    "iqr": iqr,
    "lower_bound": adr_lower,
    "upper_bound": adr_upper,
    "adr_negative_count": (df_cleaned_adr["adr"] < 0).sum(),
    "adr_above_upper_count": (df_cleaned_adr["adr"] > adr_upper).sum(),
    "adr_iqr_outlier_count": (~df_cleaned_adr["adr"].between(adr_lower, adr_upper, inclusive="both")).sum(),
})

adr_summary


q1                         69.290
q3                        126.000
iqr                        56.710
lower_bound                 0.000
upper_bound               211.065
adr_negative_count          1.000
adr_above_upper_count    3793.000
adr_iqr_outlier_count    3794.000
dtype: float64

In [74]:
# 根据 ADR 合法范围过滤异常房价记录。
adr_valid_mask = (
    (df_cleaned_adr["adr"] >= 0)
    & df_cleaned_adr["adr"].between(adr_lower, adr_upper, inclusive="both")
)

df_cleaned_adr = df_cleaned_adr.loc[adr_valid_mask].copy()

print(f"房价异常处理前行数：{df_cleaned_missing.shape[0]:,}")
print(f"房价异常处理后行数：{df_cleaned_adr.shape[0]:,}")
print(f"删除行数：{df_cleaned_missing.shape[0] - df_cleaned_adr.shape[0]:,}")


房价异常处理前行数：119,390
房价异常处理后行数：115,596
删除行数：3,794


#### 7.2.2 入住天数异常处理

本节构造 `total_stays = stays_in_weekend_nights + stays_in_week_nights` 表示总入住晚数，并使用 IQR 法则识别和筛除异常入住天数。

In [75]:
# 先合并周末和工作日入住晚数，再用 IQR 法则检查入住天数异常值。
df_cleaned_stays = df_cleaned_adr.copy()
df_cleaned_stays["total_stays"] = (
    df_cleaned_stays["stays_in_weekend_nights"]
    + df_cleaned_stays["stays_in_week_nights"]
)

stay_q1 = df_cleaned_stays["total_stays"].quantile(0.25)
stay_q3 = df_cleaned_stays["total_stays"].quantile(0.75)
stay_iqr = stay_q3 - stay_q1
stay_lower = max(0, stay_q1 - 1.5 * stay_iqr)
stay_upper = stay_q3 + 1.5 * stay_iqr

stay_summary = pd.Series({
    "q1": stay_q1,
    "q3": stay_q3,
    "iqr": stay_iqr,
    "lower_bound": stay_lower,
    "upper_bound": stay_upper,
    "total_stays_below_lower_count": (df_cleaned_stays["total_stays"] < stay_lower).sum(),
    "total_stays_above_upper_count": (df_cleaned_stays["total_stays"] > stay_upper).sum(),
    "total_stays_iqr_outlier_count": (~df_cleaned_stays["total_stays"].between(stay_lower, stay_upper, inclusive="both")).sum(),
})

stay_summary


q1                                  2.0
q3                                  4.0
iqr                                 2.0
lower_bound                         0.0
upper_bound                         7.0
total_stays_below_lower_count       0.0
total_stays_above_upper_count    4968.0
total_stays_iqr_outlier_count    4968.0
dtype: float64

In [76]:
# 根据入住天数 IQR 范围过滤极端入住时长记录。
stay_valid_mask = df_cleaned_stays["total_stays"].between(stay_lower, stay_upper, inclusive="both")
df_cleaned_anomaly = df_cleaned_stays.loc[stay_valid_mask].copy()

print(f"入住天数异常处理前行数：{df_cleaned_stays.shape[0]:,}")
print(f"入住天数异常处理后行数：{df_cleaned_anomaly.shape[0]:,}")
print(f"删除行数：{df_cleaned_stays.shape[0] - df_cleaned_anomaly.shape[0]:,}")


入住天数异常处理前行数：115,596
入住天数异常处理后行数：110,628
删除行数：4,968


### 7.3 数据去重检查

项目说明要求基于预订唯一标识去重，但当前数据集中没有发现明确的预订 ID 或订单唯一标识字段。因此本阶段无法基于唯一标识去重，只检查完全重复行。

In [ ]:
# 当前数据没有明确订单唯一标识，因此先检查完全重复行，再执行保守去重。
id_candidates = [
    col for col in df_cleaned_anomaly.columns
    if col.lower() == "id" or col.lower().endswith("_id") or "id" in col.lower()
]

duplicate_rows = df_cleaned_anomaly.duplicated().sum()

print(f"可能的 ID 字段：{id_candidates}")
print(f"完全重复行数量：{duplicate_rows:,}")

df_cleaned_dedup = df_cleaned_anomaly.drop_duplicates().copy()
print(f"去重后行数：{df_cleaned_dedup.shape[0]:,}")


### 7.4 时间格式统一

`reservation_status_date` 原始为字符串类型，本阶段将其转换为 datetime。`arrival_date` 由 `arrival_date_year`、`arrival_date_month`、`arrival_date_day_of_month` 三个字段组合生成。转换时使用 `errors="coerce"`，无法解析的日期会被转换为 `NaT`，并统计转换失败数量。

In [ ]:
# 统一日期字段格式，并由年月日字段合成标准入住日期 arrival_date。
df_cleaned_time = df_cleaned_dedup.copy()

df_cleaned_time["reservation_status_date"] = pd.to_datetime(
    df_cleaned_time["reservation_status_date"],
    errors="coerce"
)

df_cleaned_time["arrival_date"] = pd.to_datetime(
    df_cleaned_time["arrival_date_year"].astype(str)
    + "-"
    + df_cleaned_time["arrival_date_month"].astype(str)
    + "-"
    + df_cleaned_time["arrival_date_day_of_month"].astype(str),
    errors="coerce"
)

time_check = pd.Series({
    "reservation_status_date_missing_after_convert": df_cleaned_time["reservation_status_date"].isna().sum(),
    "arrival_date_missing_after_convert": df_cleaned_time["arrival_date"].isna().sum(),
})

time_check


## 8. 清洗后检查

### 8.1 清洗行数汇总

汇总原始数据、各清洗步骤后的数据规模变化。

In [ ]:
# 汇总各清洗步骤后的数据规模，便于追踪每一步删除了多少记录。
cleaning_summary = pd.Series({
    "raw_rows": df.shape[0],
    "after_missing_fill_rows": df_cleaned_missing.shape[0],
    "after_adr_cleaning_rows": df_cleaned_adr.shape[0],
    "after_stay_cleaning_rows": df_cleaned_anomaly.shape[0],
    "after_dedup_rows": df_cleaned_dedup.shape[0],
    "final_rows": df_cleaned_time.shape[0],
    "final_columns": df_cleaned_time.shape[1],
    "removed_rows_total": df.shape[0] - df_cleaned_time.shape[0],
})

cleaning_summary


In [ ]:
df_cleaned_time.head()


In [79]:
print(f"原始数据行数：{df.shape[0]:,}")
print(f"清洗后行数：{df_cleaned_time.shape[0]:,}")
print(f"清洗后列数：{df_cleaned_time.shape[1]:,}")

原始数据行数：119,390
清洗后行数：114,673
清洗后列数：38


In [80]:
pd.DataFrame({
    "missing_count": df_cleaned_time.isna().sum(),
    "missing_ratio": df_cleaned_time.isna().mean(),
}).sort_values("missing_count", ascending=False).head(15)

,missing_count,missing_ratio
hotel,0,0.0
is_canceled,0,0.0
lead_time,0,0.0
arrival_date_year,0,0.0
arrival_date_month,0,0.0
arrival_date_week_number,0,0.0
arrival_date_day_of_month,0,0.0
stays_in_weekend_nights,0,0.0
stays_in_week_nights,0,0.0
adults,0,0.0


In [81]:
df_cleaned_time["is_canceled"].value_counts(normalize=True).rename("ratio")

is_canceled
0    0.627898
1    0.372102
Name: ratio, dtype: float64

## 9. 保存清洗结果

清洗后的数据保存为新的 CSV 文件：`data/processed/hotel_bookings_cleaned.csv`。

In [ ]:
# 保存清洗后的 CSV，作为后续数据库、特征工程和 EDA 的输入。
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
df_cleaned_time.to_csv(PROCESSED_DATA_PATH, index=False, encoding="utf-8-sig")

print(f"清洗后数据：{PROCESSED_DATA_PATH}")


清洗后数据：d:\携程实习文件\hotel-booking-analysis\data\processed\hotel_bookings_cleaned.csv


## 10. 阶段说明

本 Notebook 完成了酒店预订数据的数据加载、基础质量检查、目标变量检查、字段完整性检查、多维度数据清洗、时间格式统一和清洗结果保存。

本阶段已完成的主要内容包括：

- 使用相对路径读取原始数据。
- 检查数据规模、字段类型、缺失值、唯一值数量等基础信息。
- 确认目标变量 `is_canceled` 存在，且取值为 0/1。
- 对比项目说明字段和实际字段，发现实际数据额外包含 `city` 字段。
- 对 `children`、`agent`、`company` 使用 0 进行缺失值填补。
- 使用 IQR 法则处理 `adr` 房价异常值与异常入住天数。
- 检查数据集中是否存在唯一 ID 字段；当前未发现明确预订唯一标识。
- 检查完全重复行，当前未发现完全重复行。
- 将 `reservation_status_date` 转换为 datetime。
- 根据 `arrival_date_year`、`arrival_date_month`、`arrival_date_day_of_month` 构造 `arrival_date`。
- 将清洗后的数据保存为 `data/processed/hotel_bookings_cleaned.csv`。

需要说明的问题：

- 当前数据没有明确的预订唯一 ID，因此无法基于唯一标识去重，只能进行完全重复行检查。
- `city` 字段未出现在项目说明的数据字典中，本阶段仅记录为额外字段，暂不删除。
- `reservation_status` 和 `reservation_status_date` 可能属于订单结果相关字段，后续建模时需要注意数据泄漏风险。
- `agent` 和 `company` 是匿名 ID 字段，本阶段仅做缺失值填补；是否构造缺失标记或进一步编码，留到后续特征工程阶段处理。